# Solving Time-Dependent PDEs

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differential_equations/partial_differential_equations/solving_time_dependent_pdes.ipynb)

The standard approach to solving time-dependent PDEs combines IVP time-marching with spatial BVP discretization:

1. Discretize the time derivative as an IVP relating the new state $T^{t+1}$ to the known state $T^t$.
2. Discretize the spatial derivatives on a mesh of spacing $\Delta x$ using finite differences.
3. March forward in time, enforcing boundary conditions at each time step.

Consider the 1D parabolic heat equation with thermal diffusivity $\alpha > 0$:

$$
\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}, \qquad \alpha \frac{\partial^2 T}{\partial x^2} \approx \alpha \frac{T_{i-1} - 2 T_i + T_{i+1}}{\Delta x^2}
$$

### Notation

* Spatial grid points are indexed with a subscript $i$ ($x_i = i \Delta x$).
* Discrete time levels are indexed with a superscript $t$, marching from known $T_i^t$ to unknown $T_i^{t+1}$.

In [ ]:
try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Explicit Euler Time-Stepping

Evaluating the spatial second derivative at the *current* time level $t$ yields the explicit (Forward Euler) scheme:

$$
\begin{aligned}
\frac{T^{t+1}_i - T^{t}_i}{\Delta t} &= \alpha \frac{T^t_{i-1} - 2 T^t_i + T^t_{i+1}}{\Delta x^2} \\
T^{t+1}_i &= T_i^t + r \left(T^t_{i-1} - 2 T^t_i + T^t_{i+1}\right), \qquad r = \frac{\alpha \Delta t}{\Delta x^2}
\end{aligned}
$$

![Forward Euler stencil](../images/forward_euler_stencil.svg)

Each grid point $T_i^{t+1}$ at the new time step depends *only* on three known values at time level $t$. Use the sliders below to adjust the spatial step $\Delta x$, time step $\Delta t$, and step number to cross $r = 0.5$ and see when and where the instability emerges.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def forward_euler_heat
:end-before: def backward_euler_heat
```

In [ ]:
exe.display_source(exe.forward_euler_heat)
exe.plot_forward_euler_heat()

## Von Neumann Stability Analysis

To determine when spatial oscillations grow or decay, *von Neumann stability analysis* expands the spatial profile in Fourier modes ${}^k T_i^t = e^{j k i \Delta x}$ (with wave number $k$ and $j = \sqrt{-1}$) and defines the amplification factor $G$ via ${}^k T_i^{t+1} = G \, {}^k T_i^t$. Stability requires $|G| \le 1$ for all wave numbers $k$.

Substituting a Fourier mode into the Forward Euler update gives:

$$
\begin{aligned}
{}^k T^{t+1}_i &= e^{j k i \Delta x} + r \left(e^{j k (i-1) \Delta x} - 2 e^{j k i \Delta x} + e^{j k (i+1) \Delta x}\right) \\
&= {}^k T^{t}_i \left[1 + r \left(e^{j k \Delta x} + e^{-j k \Delta x} - 2\right)\right] \\
\implies G &= 1 + 2 r \left[\cos(k \Delta x) - 1\right]
\end{aligned}
$$

Since $\cos(k \Delta x) \in [-1, 1]$, the amplification factor lies in $[1 - 4r, 1]$. Requiring $1 - 4r \ge -1$ yields the 1D stability limit (and its 2D/3D extensions on a uniform grid of spacing $h = \Delta x$):

$$
\text{1D: } \Delta t \le \frac{\Delta x^2}{2 \alpha}, \qquad \text{2D: } \Delta t \le \frac{h^2}{4 \alpha}, \qquad \text{3D: } \Delta t \le \frac{h^2}{6 \alpha}
$$

Because $\Delta t$ scales with $\Delta x^2$, halving the grid spacing forces a fourfold reduction in time step—a severe stiffness penalty shared by explicit diffusion schemes.

## Implicit Euler Time-Stepping

Evaluating the spatial finite difference at the *new* time level $t+1$ yields the Backward (Implicit) Euler scheme:

$$
-r T^{t+1}_{i-1} + (1 + 2r) T^{t+1}_i - r T^{t+1}_{i+1} = T_i^{t}
$$

![Backward Euler stencil](../images/backward_euler_stencil.svg)

Von Neumann analysis gives the amplification factor $G = \frac{1}{1 + 2r[1 - \cos(k \Delta x)]} \in \left[\frac{1}{1 + 4r}, 1\right]$, which satisfies $|G| \le 1$ for *all* $\Delta t > 0$. Backward Euler is **unconditionally stable**, at the cost of solving a tridiagonal linear system $\mathbf{A} \mathbf{T}^{t+1} = \mathbf{T}^t$ at each time step. Use the sliders below to explore large $r$ values without instability.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def backward_euler_heat
:end-before: def crank_nicolson_heat
```

In [ ]:
exe.display_source(exe.backward_euler_heat)
exe.plot_backward_euler_heat()

## Crank-Nicolson Method

Backward Euler is only first-order accurate in time ($\mathcal{O}(\Delta t)$), whereas central differencing in space is second-order ($\mathcal{O}(\Delta x^2)$). The **Crank-Nicolson method** achieves second-order accuracy in both time and space by averaging the explicit and implicit spatial discretizations at levels $t$ and $t+1$:

$$
\begin{aligned}
\frac{T^{t+1}_i - T^{t}_i}{\Delta t} &= \frac{\alpha}{2 \Delta x^2} \left[\left(T^{t+1}_{i-1} - 2 T^{t+1}_i + T^{t+1}_{i+1}\right) + \left(T^{t}_{i-1} - 2 T^{t}_i + T^{t}_{i+1}\right)\right] \\
-r T_{i-1}^{t+1} + (1 + 2r) T_i^{t+1} - r T_{i+1}^{t+1} &= r T_{i-1}^{t} + (1 - 2r) T_i^{t} + r T_{i+1}^{t}, \qquad r = \frac{\alpha \Delta t}{2 \Delta x^2}
\end{aligned}
$$

![Crank-Nicolson stencil](../images/crank_nicolson_stencil.svg)

The left-hand matrix $\mathbf{A}$ has the *exact same tridiagonal sparsity* as Backward Euler, delivering $\mathcal{O}(\Delta t^2 + \Delta x^2)$ accuracy and unconditional stability for virtually identical computational cost per step. Compare Crank-Nicolson against Backward Euler and the exact solution using the sliders below.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def crank_nicolson_heat
:end-before: crank_nicholson_heat =
```

In [ ]:
exe.display_source(exe.crank_nicolson_heat)
exe.plot_crank_nicolson_heat()